# Extract a subset of studies

The competition data is already mounted here. This copies the first `N_STUDIES` of it
into `/kaggle/working` as one archive, so it can be fetched with a **single**
`kaggle kernels output` instead of one API request per slice.

That distinction is the whole point. Asking the file endpoint for a subset costs one
request per `.dcm` — about 16,000 for 80 studies — which exhausts the account's
download quota and blocks *every* download, bulk transfers included. One archive costs
one request.

For the full corpus, do not use this: `kaggle competitions download` without `-f` is
already a single request.

Set `N_STUDIES`, run all, then locally:

```bash
kaggle kernels output mathysgouverneur/rsna-knee-extract -p /tmp/extract
tar -xf /tmp/extract/train_subset.tar -C data/raw
```

In [ ]:
import os
import shutil
import tarfile
import time
from pathlib import Path

import pandas as pd

N_STUDIES = 80
SPLIT = "train_series"

# Kaggle mounts the competition under one of these, depending on the day.
ROOT = next(p for p in (Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
                        Path("/kaggle/input/rsna-knee-abnormality-detection"))
            if (p / "test.csv").is_file())
OUT = Path("/kaggle/working")
print("competition root:", ROOT)

## What will be taken

In [ ]:
# Studies in the order the CSV lists them, so the subset is reproducible: the same
# N_STUDIES here and on any other run means the same studies.
series = pd.read_csv(ROOT / f"{SPLIT.replace('_series', '')}_series.csv",
                     dtype={"StudyInstanceUID": str, "SeriesInstanceUID": str})
studies = sorted(series["StudyInstanceUID"].unique())[:N_STUDIES]

files, size = [], 0
for study in studies:
    for path in (ROOT / SPLIT / study).rglob("*.dcm"):
        files.append(path)
        size += path.stat().st_size

print(f"{len(studies)} studies, {len(files)} slices, {size / 1024 ** 3:.1f} GB")
print(f"{size / 1024 ** 3 / len(studies):.2f} GB per study")

## Does it fit

In [ ]:
# Kaggle caps a kernel's output. Refuse rather than produce a truncated archive that
# would look like studies with missing slices — which the pipeline accepts in silence.
LIMIT_GB = 18.0
assert size / 1024 ** 3 < LIMIT_GB, (
    f"{size / 1024 ** 3:.1f} GB exceeds the {LIMIT_GB} GB budget. "
    f"Lower N_STUDIES to about {int(N_STUDIES * LIMIT_GB / (size / 1024 ** 3))}.")
print("within budget")

## Archive it

In [ ]:
# Stored, not compressed: DICOM pixel data is already compressed, so deflating it
# spends minutes to save a few per cent.
archive = OUT / "train_subset.tar"
t0 = time.time()

with tarfile.open(archive, "w") as tar:
    for i, study in enumerate(studies, 1):
        tar.add(ROOT / SPLIT / study, arcname=f"{SPLIT}/{study}")
        if i % 10 == 0 or i == len(studies):
            print(f"  {i}/{len(studies)} studies, "
                  f"{archive.stat().st_size / 1024 ** 3:.1f} GB, "
                  f"{time.time() - t0:.0f}s", flush=True)

print(f"\n{archive.name}: {archive.stat().st_size / 1024 ** 3:.2f} GB")

## Verify before downloading it

In [ ]:
# Read the archive back and count what it holds, per study. A tar that was cut short
# fails here rather than on the laptop after a long download.
with tarfile.open(archive) as tar:
    names = [m.name for m in tar.getmembers() if m.name.endswith(".dcm")]

per_study = pd.Series([n.split("/")[1] for n in names]).value_counts()
print(f"{len(names)} slices across {per_study.size} studies")
print(f"slices per study: min {per_study.min()}, median {int(per_study.median())}, "
      f"max {per_study.max()}")

assert len(names) == len(files), f"archive holds {len(names)} of {len(files)} slices"
assert per_study.size == len(studies), "some study is missing entirely"
print("\ncomplete")